Name: Sahil Ghogare

RollNo: 23102A0061

Experiment no.:5

In [ ]:
import pandas as pd
import pickle

# load model
with open("heart_risk_model.pkl", "rb") as f:
    model = pickle.load(f)

# load dataset
df = pd.read_csv("/content/heart_attack_prediction_india.csv")

# Preprocessing steps to match model's expected features
# Drop columns not used in training or that are identifiers
X = df.drop(columns=["Heart_Attack_Risk", "Patient_ID", "Emergency_Response_Time"])
y = df["Heart_Attack_Risk"]

# Map 'Gender' to numerical values (0 for Female, 1 for Male)
X['Gender'] = X['Gender'].map({'Female': 0, 'Male': 1})

# One-hot encode 'State_Name'
X = pd.get_dummies(X, columns=['State_Name'], drop_first=False)

# Align columns with the model's expected features
# This is crucial to handle cases where new states appear or some states are missing compared to training data
if hasattr(model, 'feature_names_in_'):
    model_features = list(model.feature_names_in_)
    # Add missing columns to X, filling with 0
    missing_cols = set(model_features) - set(X.columns)
    for c in missing_cols:
        X[c] = 0
    # Drop columns from X that are not in model_features
    extra_cols = set(X.columns) - set(model_features)
    X = X.drop(columns=list(extra_cols))
    # Ensure the order of columns is the same as in model_features
    X = X[model_features]
else:
    print("Warning: Model does not have 'feature_names_in_' attribute. Feature alignment might be incomplete.")

# sensitive attribute (0 = female, 1 = male)
sensitive = df["Gender"].map({'Female': 0, 'Male': 1})

In [ ]:
y_pred = model.predict(X)

In [ ]:
from fairlearn.metrics import MetricFrame
from sklearn.metrics import accuracy_score, recall_score

metrics = {
    "accuracy": accuracy_score,
    "recall": recall_score
}

metric_frame = MetricFrame(
    metrics=metrics,
    y_true=y,
    y_pred=y_pred,
    sensitive_features=sensitive
)

print(metric_frame.by_group)

        accuracy  recall
Gender                  
0       0.296610     1.0
1       0.304025     1.0


In [ ]:
from fairlearn.metrics import demographic_parity_difference

dp = demographic_parity_difference(y, y_pred, sensitive_features=sensitive)

print("Demographic Parity Difference:", dp)

Demographic Parity Difference: 0.0


In [ ]:
from fairlearn.metrics import equal_opportunity_difference

eo = equal_opportunity_difference(y, y_pred, sensitive_features=sensitive)

print("Equal Opportunity Difference:", eo)

Equal Opportunity Difference: 0.0


In [ ]:
from fairlearn.metrics import equalized_odds_difference

eod = equalized_odds_difference(y, y_pred, sensitive_features=sensitive)

print("Equalized Odds Difference:", eod)

Equalized Odds Difference: 0.0


In [ ]:
import numpy as np

group0 = y_pred[sensitive == 0]
group1 = y_pred[sensitive == 1]

di = np.mean(group0) / np.mean(group1)

print("Disparate Impact:", di)

Disparate Impact: 1.0


### Explanation of Fairness Metric Outputs

Let's interpret the results from the fairness metrics we've calculated:

1.  **MetricFrame (Accuracy and Recall by Gender):**
    *   **Output:**
        ```
        accuracy  recall
        Gender                  
        0       0.296610     1.0
        1       0.304025     1.0
        ```
    *   **Interpretation:** This table shows the `accuracy` and `recall` scores for each group within the sensitive attribute ('Gender').
        *   For `Gender` 0 (Female), the accuracy is approximately 29.66%, and recall is 100%.
        *   For `Gender` 1 (Male), the accuracy is approximately 30.40%, and recall is 100%.
        *   **Note:** The high recall (1.0 for both groups) coupled with low accuracy (around 0.3) suggests that the model is predicting 'Heart Attack Risk' for almost everyone, which leads to high recall for the positive class (all actual positive cases are correctly identified because nearly all cases are predicted as positive). This usually indicates a heavily imbalanced dataset or a model that is over-predicting the positive class. However, in terms of fairness, the recall is equal between groups.

2.  **Demographic Parity Difference:**
    *   **Output:** `Demographic Parity Difference: 0.0`
    *   **Interpretation:** Demographic parity requires that the proportion of individuals receiving a positive outcome (e.g., being predicted as having Heart Attack Risk) is approximately equal across different groups of the sensitive attribute. A value of `0.0` indicates perfect demographic parity, meaning the model predicts a heart attack risk for both gender groups at the same rate. This is generally considered a strong indicator of fairness under this definition.

3.  **Equal Opportunity Difference:**
    *   **Output:** `Equal Opportunity Difference: 0.0`
    *   **Interpretation:** Equal opportunity focuses on ensuring that groups have equal true positive rates (recall) when the outcome is positive. A value of `0.0` means the model correctly identifies individuals who *actually* have heart attack risk at the same rate for both gender groups. This aligns with the recall values of 1.0 for both groups, indicating that if someone truly has a heart attack risk, the model is equally likely to predict it for both genders.

4.  **Equalized Odds Difference:**
    *   **Output:** `Equalized Odds Difference: 0.0`
    *   **Interpretation:** Equalized odds is a stricter criterion than equal opportunity, requiring that both true positive rates (recall) and false positive rates are equal across sensitive groups. A value of `0.0` suggests that the model's performance is balanced for both actual positive and actual negative cases across the gender groups. Since both equal opportunity (TPR) and demographic parity (overall positive rate) are 0.0, it is not surprising that equalized odds is also 0.0.

5.  **Disparate Impact:**
    *   **Output:** `Disparate Impact: 1.0`
    *   **Interpretation:** Disparate impact is calculated as the ratio of the positive prediction rate of the unprivileged group to that of the privileged group. A value of `1.0` signifies that there is no disparate impact, meaning the model's positive prediction rate is identical for both gender groups. Values significantly deviating from 1 (e.g., typically outside the range of 0.8 to 1.25) would indicate potential bias.

In summary, based on these metrics, the model appears to be highly fair with respect to the 'Gender' attribute, as all measured differences are zero or the ratio is one. However, the high recall and low accuracy observed in the MetricFrame suggest that while the model is fair, its overall predictive performance (specifically in distinguishing negative cases) might warrant further investigation.